### Making the 735 kv network converge

In [4]:
%cd ~/Documents/McGill/capstone/quebec-pypsa-earth

/home/etienne/Documents/McGill/capstone/quebec-pypsa-earth


In [ ]:
!python network/run_pf.py --network network/networks_current/elec_solved.nc --method pf --pv-min-capacity 0 --pv-max-load-ratio 1

!mv network/networks_current/elec_solved_pf.nc etienne_tests/results/elec_315kv_pf.nc

/home/etienne/Documents/McGill/capstone/quebec-pypsa-earth
Loading solved network: network/networks_current/elec_solved.nc
INFO:pypsa.io:Imported network elec_solved.nc has buses, carriers, generators, lines, links, loads, storage_units, transformers
  205 buses, 168 snapshots
  Copied LOPF dispatch (generators.p) into p_set
  Copied LOPF dispatch (storage_units.p) into p_set
  Copied LOPF dispatch (links.p0) into p_set

Sub-networks:
attribute  carrier slack_bus           obj
SubNetwork                                
0               AC       339  SubNetwork 0
1               DC        65  SubNetwork 1
2               DC       161  SubNetwork 2
3               DC      1015  SubNetwork 3
4               DC      2206  SubNetwork 4
5               DC      3549  SubNetwork 5
  SubNetwork 0 (AC, 200 buses): slack_bus=339, slack generator(s)=['339 slack-placeholder']
  SubNetwork 1 (DC, 1 buses): slack_bus=65, slack generator(s)=[]
  SubNetwork 2 (DC, 1 buses): slack_bus=161, slack generato

What happens when we run AC power flow on our 315kv network?  

Input: network/networks_current/elec_solved.nc  
Output: looking at v_mag_pu, and converging snapshots (3/168) clearly the model does not converge. We also observe load shedding.  

What happens if we apply the fixes that we applied to the 735kV network to this one?



In [22]:
!python network/correct_sending_end_circuits.py --network network/networks_current/elec_solved.nc --output etienne_tests/results/elec_315kv.nc
!python network/run_pf.py --network etienne_tests/results/elec_315kv.nc --method pf --pv-min-capacity 0 --pv-max-load-ratio 1

Loading network: network/networks_current/elec_solved.nc
INFO:pypsa.io:Imported network elec_solved.nc has buses, carriers, generators, lines, links, loads, storage_units, transformers
Corrected circuit count on 9 sending-end lines:
      num_parallel_old  r_old   x_old  b_old  s_nom_old  num_parallel_new  r_new   x_new  b_new  s_nom_new
Line                                                                                                      
2408               1.0   5.01  165.21    0.0    5242.78               3.0   1.67   55.07   0.01   15728.35
162                1.0   4.57  150.67    0.0    5682.19               3.0   1.52   50.22   0.01   17046.58
3756               1.0   6.23  205.45    0.0    4380.16               3.0   2.08   68.48   0.01   13140.49
1049               1.0   5.63  185.91    0.0    4746.22               1.5   3.76  123.94   0.00    7119.33
1448               1.0   5.68  187.35    0.0    4716.22               1.5   3.78  124.90   0.00    7074.33
156               

Okay, this is better. It still doesn't converge well, but we have convergence for 10/168 snapshots, which is better than 3. Let's continue.

In [9]:
!python network/apply_series_compensation.py --network etienne_tests/results/elec_315kv.nc --in-place
!python network/run_pf.py --network etienne_tests/results/elec_315kv.nc --method pf --pv-min-capacity 0 --pv-max-load-ratio 1

Loading network: etienne_tests/results/elec_315kv.nc
INFO:pypsa.io:Imported network elec_315kv.nc has buses, carriers, generators, lines, links, loads, storage_units, transformers
Applied 70% series compensation to 15 lines:
  2408 (114-1291 (500.6km, sole supply path to bus 1291)): x 16.52 -> 16.52 ohm
  880 (312-308 circuit 1 (433.4km)): x 42.91 -> 42.91 ohm
  1935 (312-308 circuit 2 (409.5km)): x 40.54 -> 40.54 ohm
  2410 (312-308 circuit 3 (409.6km)): x 40.55 -> 40.55 ohm
  1536 (308-2944 (256.2km)): x 25.36 -> 25.36 ohm
  1538 (308-1081 (266.1km)): x 26.34 -> 26.34 ohm
  1936 (308-136 (348.0km)): x 34.45 -> 34.45 ohm
  163 (457-312 circuit 1 (344.7km)): x 34.12 -> 34.12 ohm
  1655 (457-312 circuit 2 (344.6km)): x 34.12 -> 34.12 ohm
  2386 (457-312 circuit 3 (378.2km)): x 37.44 -> 37.44 ohm
  162 (340-457 (456.6km, highest transmission angle on the network)): x 15.07 -> 15.07 ohm
  156 (66-457 circuit 1 (432.7km, high transmission angle)): x 28.56 -> 28.56 ohm
  1938 (66-457 circui

Applying series compensation gives convergence of 106/168 snapshots. The voltages look off though, they are way too high.

**The following codeblock takes a while to execute**

In [11]:
!python network/add_shunt_impedance.py --network etienne_tests/results/elec_315kv.nc --output etienne_tests/results/elec_315kv_shunt.nc
!python network/run_pf.py --network etienne_tests/results/elec_315kv_shunt.nc --method pf --pv-min-capacity 0 --pv-max-load-ratio 1

INFO:pypsa.io:Imported network elec_315kv.nc has buses, carriers, generators, lines, links, loads, storage_units, transformers

--- Preparing for AC PF (no-op for DC/lpf) ---
1. Transformer r = x / 30 (generic X/R, not measured) on 22 transformers
2. Load q_set = p_set * tan(acos(0.95)) on 647 loads (generic 0.95 lagging power factor assumption)
1b. Added 14 zero-dispatch placeholder generator(s) on storage-only buses (20528 MW combined, sized to match), so they're eligible for PV classification below
2b. Generator q_set = p_nom * tan(acos(0.9)) on 87 real generators (generic 0.9 PF reactive capability, fixed to installed capacity not dispatch)
3. Marked 45 / 52 generator buses PV (p_nom >= 0 MW and load < 100% of local generation; largest plain Generator there each -- including the storage-pv-placeholders added in step 1b, now eligible on the same terms as real generators). Remaining generator buses stay PQ (too small and/or too load-dominated to plausibly regulate voltage in practice

This one seems to not work at all. The code takes a really long time to finish running, with many warnings about too many iterations.

In [15]:
import numpy as np, pandas as pd, pypsa
lpf = pypsa.Network("network/networks_current/elec_solved_lpf.nc")
va = lpf.buses_t.v_ang
ang = np.degrees(va[lpf.lines.bus0].values - va[lpf.lines.bus1].values)
S = pd.DataFrame({
    "v_nom": lpf.lines.v_nom, "length": lpf.lines.length, "circuits": lpf.lines.num_parallel,
    "max_angle": pd.Series(np.abs(ang).max(axis=0), index=lpf.lines.index),
    "max_loading": lpf.lines_t.p0.abs().max() / lpf.lines.s_nom,
})
S["km_per_circuit"] = S.length / S.circuits
print(S.sort_values("max_angle", ascending=False).head(25).to_string())

INFO:pypsa.io:Imported network elec_solved_lpf.nc has buses, carriers, generators, lines, links, loads, storage_units, transformers


      v_nom      length  circuits  max_angle  max_loading  km_per_circuit
Line                                                                     
3360  315.0  409.614824       1.0  59.246858     0.661404      409.614824
3756  735.0  622.590677       1.0  51.155329     0.535965      622.590677
1049  735.0  563.370794       1.0  49.096465     0.524621      563.370794
1448  735.0  567.736705       1.0  49.096465     0.523898      567.736705
3352  315.0  182.112092       1.0  44.179415     0.510816      182.112092
2408  735.0  500.629917       1.0  43.606246     0.474688      500.629917
3535  735.0  471.427214       1.0  40.776823     0.447387      471.427214
2410  735.0  409.550614       1.0  39.484447     0.439759      409.550614
1935  735.0  409.501021       1.0  39.484447     0.439763      409.501021
880   735.0  433.409000       1.0  39.484447     0.437335      433.409000
162   735.0  456.561283       1.0  35.569285     0.391740      456.561283
156   735.0  432.670772       1.0  34.

In [18]:
m = pypsa.Network("network/networks_current/elec_solved.nc")
lid, k = "3352", 2
m.lines.loc[lid, ["r", "x"]] /= k
m.lines.loc[lid, ["b", "s_nom", "num_parallel"]] *= k

INFO:pypsa.io:Imported network elec_solved.nc has buses, carriers, generators, lines, links, loads, storage_units, transformers


In [19]:
!python network/run_pf.py --network network/networks_current/elec_solved.nc --method pf --pv-min-capacity 0 --pv-max-load-ratio 1

Loading solved network: network/networks_current/elec_solved.nc
INFO:pypsa.io:Imported network elec_solved.nc has buses, carriers, generators, lines, links, loads, storage_units, transformers
  205 buses, 168 snapshots
  Copied LOPF dispatch (generators.p) into p_set
  Copied LOPF dispatch (storage_units.p) into p_set
  Copied LOPF dispatch (links.p0) into p_set

Sub-networks:
attribute  carrier slack_bus           obj
SubNetwork                                
0               AC       339  SubNetwork 0
1               DC        65  SubNetwork 1
2               DC       161  SubNetwork 2
3               DC      1015  SubNetwork 3
4               DC      2206  SubNetwork 4
5               DC      3549  SubNetwork 5
  SubNetwork 0 (AC, 200 buses): slack_bus=339, slack generator(s)=['339 slack-placeholder']
  SubNetwork 1 (DC, 1 buses): slack_bus=65, slack generator(s)=[]
  SubNetwork 2 (DC, 1 buses): slack_bus=161, slack generator(s)=['161 load_shedding']
  SubNetwork 3 (DC, 1 buses): sl

Adding circuits for the two long 315 kV lines does not affect the convergence.